# 03. Stage 2 resilience measures and regularized predictive models

Combines the 2018 OMB Stage 1 outcome and pre-February-2020 predictor datasets. Predictive features use only 2018 ACS five-year and 2018–2019 Q2 QCEW information that was eligible for public availability as of February 2020. Post-March-2020 outcomes *are targets only*.

The four principal resilience measures are: (1) 2020 resistance shortfall, (2) recovery gain relative to the 2020 trough, (3) late-2023 persistence shortfall, and (4) cumulative positive employment deficit as percent-point-month area. Shortfall metrics are *higher = worse*, recovery gain is *higher = better*. Recovery status requires three consecutive months at or above counterfactual; never-recovered regions are right-censored, not assigned invented times.

In [1]:
from pathlib import Path
import os,json,hashlib,warnings
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold,GroupShuffleSplit,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge,Lasso,ElasticNet
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
ROOT=Path(os.environ['RL_PROJECT_ROOT']).resolve() if os.environ.get('RL_PROJECT_ROOT') else next(
 (p for p in [Path.cwd(),*Path.cwd().parents] if (p/'output'/'stage1_omb2018_manifest.json').exists()),None)
if ROOT is None:raise FileNotFoundError('Run notebooks 00–02 first')
PROC=ROOT/'data'/'processed';OUT=ROOT/'output'
GEO=json.loads((PROC/'omb2018_geography_manifest.json').read_text())
STAGE1=json.loads((OUT/'stage1_omb2018_manifest.json').read_text())
PREDMETA=json.loads((OUT/'stage2_omb2018_predictors_manifest.json').read_text())
if {GEO['geography_vintage'],STAGE1['geography_vintage'],PREDMETA['geography_vintage']}!={'2018-09'}:
 raise ValueError('Cross-vintage merge prohibited')
if STAGE1['model_training_cutoff']!='2019-06-01':raise ValueError('Stage 1 uses post-cutoff training')
if PREDMETA['information_cutoff']!='2020-02-29':raise ValueError('Predictors use incorrect cutoff')
def sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for x in iter(lambda:f.read(4*1024*1024),b''):h.update(x)
 return h.hexdigest()
DATA=PROC/'stage1_omb2018_observed_vs_counterfactual_2020_2023.csv'
PRED=PROC/'stage2_omb2018_pre2020_predictors.csv'
if sha256(DATA)!=STAGE1['outcomes_sha256'] or sha256(PRED)!=PREDMETA['predictors_sha256']:
 raise ValueError('Stage inputs changed since their provenance manifests')

## 1. Resilience outcomes from the 46 post-March-2020 months

In [2]:
gaps=pd.read_csv(DATA,dtype={'cbsa_code':'string'},parse_dates=['date'])
gaps.cbsa_code=gaps.cbsa_code.str.zfill(5)
if gaps.duplicated(['cbsa_code','date']).any():raise ValueError('Duplicate month rows')
if not gaps.groupby('cbsa_code').size().eq(46).all():raise ValueError('Incomplete 46-month evaluation')
if gaps.date.min()!=pd.Timestamp('2020-03-01') or gaps.date.max()!=pd.Timestamp('2023-12-01'):
 raise ValueError('Unexpected evaluation dates')
if (gaps.counterfactual_employment<=0).any():raise ValueError('Nonpositive counterfactual employment')
gaps['gap_pct']=100*(gaps.employment/gaps.counterfactual_employment-1)
gaps['shortfall_pct']=(-gaps.gap_pct).clip(lower=0)
gaps.to_csv(PROC/'stage2_omb2018_monthly_gaps.csv',index=False)

def measure(s):
 s=s.sort_values('date').reset_index(drop=True)
 q=lambda start,end:s.loc[s.date.between(start,end),'gap_pct']
 first=q('2020-04-01','2020-06-01')
 trough=s.loc[s.date.between('2020-04-01','2020-12-01'),'gap_pct'].min()
 last=q('2023-10-01','2023-12-01')
 above=s.gap_pct.ge(0).to_numpy()
 recover=next((i-1 for i in range(2,len(above)) if above[i-2:i+1].all()),None)
 return pd.Series({
  'resistance_shortfall_pct_aprjun2020':float((-first).clip(lower=0).mean()),
  'initial_2020_trough_gap_pct':float(trough),
  'recovery_gain_from_trough_pp':float(last.mean()-trough),
  'persistence_shortfall_pct_octdec2023':float((-last).clip(lower=0).mean()),
  'cumulative_shortfall_12_pp_months':float(s.shortfall_pct.iloc[:12].sum()),
  'cumulative_shortfall_24_pp_months':float(s.shortfall_pct.iloc[:24].sum()),
  'cumulative_shortfall_46_pp_months':float(s.shortfall_pct.sum()),
  'recovered_by_dec2023':int(recover is not None),
  'months_to_sustained_recovery':float(recover+1) if recover is not None else np.nan,
  'gap_pct_dec2023':float(s.gap_pct.iloc[-1]),
  'employment_2020apr':float(s.loc[s.date.eq(pd.Timestamp('2020-04-01')),'employment'].iloc[0])})
rows=[]
for cbsa,s in gaps.groupby('cbsa_code',sort=True):
 r=measure(s);r['cbsa_code']=cbsa;rows.append(r)
outcomes=pd.DataFrame(rows)
cols=['cbsa_code']+[x for x in outcomes if x!='cbsa_code']
outcomes=outcomes[cols]
outcomes.to_csv(PROC/'stage2_omb2018_resilience_outcomes.csv',index=False)
print('CBSA-level outcomes:',len(outcomes))

CBSA-level outcomes: 936


## 2. Join predictors with strict vintage and sample validation

In [3]:
pred=pd.read_csv(PRED,dtype={'cbsa_code':'string'})
pred['cbsa_code']=pred.cbsa_code.str.zfill(5)
if pred.cbsa_code.duplicated().any():raise ValueError('Duplicate predictors')
merged=outcomes.merge(pred,on='cbsa_code',how='inner',validate='one_to_one')
if len(merged)!=len(outcomes):raise ValueError('Missing regional predictors in Stage 2')
merged.to_csv(PROC/'stage2_omb2018_modeling_table.csv',index=False)
print('Matched regions:',len(merged),'Geographic groups:',merged.geographic_group.nunique())

Matched regions: 936 Geographic groups: 10


## 3. Ridge, Lasso, Elastic Net with geographically held-out validation

A 20% held-out *Census-division group* split is used for final evaluation; inner GroupKFold tunes hyperparameters using training groups only. Imputation and scaling occur inside CV folds. Variable filtering by availability is based only on the training data. Models are **predictive associations**, not causal effects.

In [4]:
TARGETS=[
 'resistance_shortfall_pct_aprjun2020',
 'recovery_gain_from_trough_pp',
 'persistence_shortfall_pct_octdec2023',
 'cumulative_shortfall_46_pp_months']
EXCLUDED={'cbsa_code','geographic_group','cbsa_type'}
FEATURES=[c for c in pred if c not in EXCLUDED and pd.api.types.is_numeric_dtype(pred[c])]
if len(FEATURES)<2:raise ValueError('Insufficient numerical predictors')
if len(merged)<20 or merged.geographic_group.nunique()<4:
 print('Small sample: skip Stage 2B modeling; outcomes were saved. Need ≥20 CBSAs and ≥4 groups.')
else:
 groups=merged.geographic_group.fillna('Unknown').astype(str)
 train_i,test_i=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(
  merged,groups=groups))
 test_groups=set(groups.iloc[test_i]);train_groups=set(groups.iloc[train_i])
 if not test_groups.isdisjoint(train_groups):raise RuntimeError('Geographic holdout leakage')
 train=merged.iloc[train_i];test=merged.iloc[test_i]
 available=train[FEATURES].notna().mean()
 features=available[available>=0.50].index.tolist()
 if len(features)<2:raise ValueError('Fewer than two 50%-covered numeric training predictors')
 cv=GroupKFold(n_splits=min(5,train.geographic_group.nunique()))
 # These are regularized linear models, fitted with training-only transformations.
 methods={
  'ridge':(Ridge(max_iter=10000),{'model__alpha':[.01,.1,1,10,100,1000]}),
  'lasso':(Lasso(max_iter=20000,random_state=2026),{'model__alpha':[.001,.01,.1,1,10,100]}),
  'elasticnet':(ElasticNet(max_iter=20000,random_state=2026),
                {'model__alpha':[.001,.01,.1,1,10,100],
                 'model__l1_ratio':[.2,.5,.8]})}
 scores=[];test_predictions=[];coefs=[]
 for target in TARGETS:
  yt=train[target].to_numpy(float);ye=test[target].to_numpy(float)
  for method,(est,params) in methods.items():
   pipe=Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),
                  ('scale',StandardScaler()),('model',est)])
   search=GridSearchCV(pipe,params,cv=cv,scoring='neg_mean_absolute_error',n_jobs=1,
                       error_score='raise')
   with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    search.fit(train[features],yt,groups=train.geographic_group)
   predicted=search.predict(test[features])
   metric={'outcome':target,'model':method,'n_train':len(train),'n_test':len(test),
           'n_features':len(features),'mae':mean_absolute_error(ye,predicted),
           'rmse':float(np.sqrt(mean_squared_error(ye,predicted))),
           'r2':float(r2_score(ye,predicted)),
           'baseline_train_mean_mae':float(np.mean(np.abs(ye-yt.mean()))),
           'inner_cv_mae':-float(search.best_score_),
           'best_params':json.dumps(search.best_params_),
           'held_out_groups':','.join(sorted(test_groups))}
   scores.append(metric)
   for cbsa,actual,estimate in zip(test.cbsa_code,ye,predicted):
    test_predictions.append({'cbsa_code':cbsa,'outcome':target,'model':method,
                             'observed':float(actual),'predicted':float(estimate)})
   coefs.extend({'outcome':target,'model':method,'feature':feature,'standardized_coef':float(v)}
                for feature,v in zip(features,search.best_estimator_.named_steps['model'].coef_))
   print(target,method,'held-out MAE:',round(metric['mae'],3))
 pd.DataFrame(scores).to_csv(OUT/'stage2_omb2018_predictive_model_scores.csv',index=False)
 pd.DataFrame(test_predictions).to_csv(OUT/'stage2_omb2018_test_predictions.csv',index=False)
 pd.DataFrame(coefs).to_csv(OUT/'stage2_omb2018_standardized_coefficients.csv',index=False)
 pd.DataFrame({'train_cbsa':train.cbsa_code}).to_csv(OUT/'stage2_omb2018_training_cbsa.csv',index=False)
 pd.DataFrame({'test_cbsa':test.cbsa_code}).to_csv(OUT/'stage2_omb2018_heldout_cbsa.csv',index=False)
 print('Saved predictive model scores and test-set forecasts')

resistance_shortfall_pct_aprjun2020 ridge held-out MAE: 3.558
resistance_shortfall_pct_aprjun2020 lasso held-out MAE: 3.645
resistance_shortfall_pct_aprjun2020 elasticnet held-out MAE: 3.548
recovery_gain_from_trough_pp ridge held-out MAE: 5.274
recovery_gain_from_trough_pp lasso held-out MAE: 5.348
recovery_gain_from_trough_pp elasticnet held-out MAE: 5.348
persistence_shortfall_pct_octdec2023 ridge held-out MAE: 2.763
persistence_shortfall_pct_octdec2023 lasso held-out MAE: 2.762
persistence_shortfall_pct_octdec2023 elasticnet held-out MAE: 2.762
cumulative_shortfall_46_pp_months ridge held-out MAE: 112.554
cumulative_shortfall_46_pp_months lasso held-out MAE: 112.578
cumulative_shortfall_46_pp_months elasticnet held-out MAE: 112.62
Saved predictive model scores and test-set forecasts


## 4. Reproducibility and limitations

In [5]:
manifest3={'geography_vintage':'2018-09','information_cutoff':'2020-02-29',
 'forecast_training_cutoff':'2019-06-01',
 'outcome_window':'2020-03 through 2023-12',
 'modeling_cbsa_count':len(merged),
 'predictors_sha256':sha256(PRED),'outcomes_sha256':sha256(DATA),
 'limitations':[
  'Later downloaded QCEW archives may include revisions not published by February 2020.',
  'Counterfactuals are extrapolations, not identified causal COVID effects.',
  'Missing QCEW industry shares are left missing upstream; training-only imputation used.',
  'Recovery duration is right-censored when not achieved by December 2023.',
  '2018 OMB delineations held constant across all years.']}
(OUT/'stage2_omb2018_modeling_manifest.json').write_text(json.dumps(manifest3,indent=2))
print('Stage 2 complete. Inspect coverage, data-vintage, model and geographic audits.')

Stage 2 complete. Inspect coverage, data-vintage, model and geographic audits.
